In [7]:
print("Hello World")

Hello World


### Step 1: Parsing the XML files to extract data from them. It is extracted in the form of key value pairs the key being section titles/headers and the values is the corresponding section text. 

In [8]:
from bs4 import BeautifulSoup
import pandas as pd
from pathlib import Path

def extract_headers_and_sections_from_xml(xml_path, wanted_titles=None):
    """
    Extracts headers and their corresponding section texts from a Rechtspraak XML file.

    Returns:
        headers: list of section titles
        sections: list of dicts: {"title": title, "text": section_text}
    """
    with open(xml_path, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "xml")

    sections = []

    for sec in soup.find_all(lambda t: t.name and t.name.lower().endswith("section")):
        title_tag = sec.find(lambda t: t.name and t.name.lower().endswith("title"))

        nr_tag = (
            title_tag.find(lambda t: t.name and t.name.lower().endswith("nr"))
            if title_tag else None
        )

        nr = nr_tag.text.strip() if nr_tag else ""
        title_text = "".join(title_tag.stripped_strings) if title_tag else ""

        if nr:
            title_text = title_text.replace(nr, "", 1).strip()

        section_text = " ".join(
            [s for s in sec.stripped_strings if s not in (nr, title_text)]
        )

        sections.append({
            "title": title_text,
            "text": section_text
        })

    if wanted_titles:
        wanted_set = {t.strip().lower() for t in wanted_titles}

        headers = [
            s["title"] for s in sections
            if s["title"].strip().lower() in wanted_set
        ]

        filtered_sections = [
            s for s in sections
            if s["title"].strip().lower() in wanted_set
        ]
    else:
        headers = [s["title"] for s in sections if s["title"]]
        filtered_sections = [s for s in sections if s["title"]]

    return headers, filtered_sections


# ===========================
# BATCH EXTRACTION FROM XML FOLDER
# ===========================

xml_folder = Path("rechtspraak_xml")

data = []

# Get every XML file in the folder
xml_files = sorted(xml_folder.glob("*.xml"))

for xml_path in xml_files:
    case_name = xml_path.stem  # filename without .xml

    try:
        headers, sections = extract_headers_and_sections_from_xml(xml_path)

        # Convert to dict: {title: text}
        title_text_dict = {
            s["title"]: s["text"]
            for s in sections
            if s.get("title") and s.get("text")
        }

        data.append({
            "case_name": case_name,
            "xml_file": xml_path.name,
            "extracted_data": title_text_dict
        })

    except Exception as e:
        data.append({
            "case_name": case_name,
            "xml_file": xml_path.name,
            "extracted_data": None,
            "error": str(e)
        })

# Build final DataFrame
df_final = pd.DataFrame(data)

# Keep only rows with non-empty extracted dictionaries
df_final = df_final[
    df_final["extracted_data"].apply(lambda d: isinstance(d, dict) and len(d) != 0)
].reset_index(drop=True)

df_final

,case_name,xml_file,extracted_data
0,ECLI_NL_GHAMS_2015_2960,ECLI_NL_GHAMS_2015_2960.xml,{'Het geding in hoger beroep': 'Partijen worde...
1,ECLI_NL_GHAMS_2015_5282,ECLI_NL_GHAMS_2015_5282.xml,{'Het verloop van het geding': '1.1 In het ver...
2,ECLI_NL_GHAMS_2016_4005,ECLI_NL_GHAMS_2016_4005.xml,{'Het geding in hoger beroep': '1.1. Appellant...
3,ECLI_NL_GHAMS_2017_3990,ECLI_NL_GHAMS_2017_3990.xml,{'Ontstaan en loop van het geding': '1.1. De h...
4,ECLI_NL_GHAMS_2017_4813,ECLI_NL_GHAMS_2017_4813.xml,{'Onderzoek van de zaak': 'Dit arrest is gewez...
...,...,...,...
95,ECLI_NL_RBZWB_2015_638,ECLI_NL_RBZWB_2015_638.xml,{'Procesverloop': 'Eiser heeft beroep ingestel...
96,ECLI_NL_RBZWB_2015_6953,ECLI_NL_RBZWB_2015_6953.xml,{'De procedure': '1.1. Het verloop van de proc...
97,ECLI_NL_RBZWB_2019_5711,ECLI_NL_RBZWB_2019_5711.xml,{'Onderzoek van de zaak': 'De zaak is inhoudel...
98,ECLI_NL_RBZWB_2020_2644,ECLI_NL_RBZWB_2020_2644.xml,"{'de gemeente GEMEENTE NOORD-BEVELAND,': 'geve..."


### Step 2 - Removing certain unwanted sections, which are not a part of the annotation scheme and would just introduce noise.

This works based on certain seed words which refer to the section titles. Those to be removed are matched and removed. 

In [9]:
## Remove unwanted titles and their sections from the extracted data ##

# Define title groups (section titles to remove)
# Each group contains possible variants/synonyms of a section title
title_groups = {
    "wettelijk_kader": [
        "wettelijk_kader", "toepasselijke wettelijke voorschriften", "de wettelijke voorschriften",
        "toepassende wetsbepalingen", "toepasselijke wetsartikelen", "de toegepaste wettelijke voorschriften",
        "de toepasselijke wetsartikelen", "de toegepaste wettelijke bepalingen"
    ],
    "proceskosten": [
        "proceskosten", "griffierecht en proceskosten", "kosten", "proceskosten en griffierecht"
    ],
    "procesverloop": 
    [
        "procesverloop", "de procedure", "het geding in hoger beroep", "het procesverloop",
        "het geding in eerste aanleg", "ontstaan en loop van het geding", "procesgang",
        "procesverloop in cassatie", "het verloop van de procedure", "procesverloop",
        "het verloop van het geding in eerste aanleg", "het verdere verloop van het geding in hoger beroep",
        "het verloop van het geding", "het verloop van de procedure in hoger beroep",
        "de procedure in hoger beroep", "procesverloop in hoger beroep", "procedure",
        "het verdere verloop van de procedure in hoger beroep", "het verdere verloop van de procedure",
        "de procedure bij de rechtbank", "de procedure in eerste aanleg", "verloop van de procedure",
        "het verdere procesverloop", "procedure bij de rechtbank", "het geschil en de beslissing in eerste aanleg"
    ],
}

import re

def normalize(s):
    """Normalize a title: collapse whitespace, strip and lowercase."""
    return re.sub(r"\s+", " ", (s or "")).strip().lower()

# Flatten to a set of normalized titles to remove
titles_to_remove = set(normalize(t) for vals in title_groups.values() for t in vals)
print(titles_to_remove)

### Remove specified titles from extracted data ###
list_removed = []
list_newd = []
for i,extracted_items in enumerate(df_final.extracted_data,start=1): 
    titles = list(extracted_items.keys())
    n_titles = [normalize(t) for t in titles]
    removed = {}
    for t in n_titles:
        if t in titles_to_remove:
            print(f"Case {i} - Will remove: {t}")
            removed[t] = extracted_items[titles[n_titles.index(t)]]
            extracted_items.pop(titles[n_titles.index(t)])
    list_removed.append(removed)
    list_newd.append(extracted_items if extracted_items else None)
df_final['removed_titles'] = list_removed
df_final

{'toepasselijke wetsartikelen', 'de procedure', 'verloop van de procedure', 'het verdere procesverloop', 'het verdere verloop van de procedure in hoger beroep', 'procesverloop', 'wettelijk_kader', 'het geding in eerste aanleg', 'toepasselijke wettelijke voorschriften', 'procesverloop in cassatie', 'de procedure in eerste aanleg', 'de wettelijke voorschriften', 'de procedure bij de rechtbank', 'het geschil en de beslissing in eerste aanleg', 'het verloop van de procedure', 'het verdere verloop van het geding in hoger beroep', 'procesgang', 'kosten', 'het verdere verloop van de procedure', 'proceskosten', 'proceskosten en griffierecht', 'de toegepaste wettelijke voorschriften', 'de procedure in hoger beroep', 'toepassende wetsbepalingen', 'het verloop van de procedure in hoger beroep', 'procedure', 'ontstaan en loop van het geding', 'procedure bij de rechtbank', 'het verloop van het geding', 'het geding in hoger beroep', 'het verloop van het geding in eerste aanleg', 'de toegepaste wette

,case_name,xml_file,extracted_data,removed_titles
0,ECLI_NL_GHAMS_2015_2960,ECLI_NL_GHAMS_2015_2960.xml,{'Feiten': 'De kantonrechter heeft in het best...,{'het geding in hoger beroep': 'Partijen worde...
1,ECLI_NL_GHAMS_2015_5282,ECLI_NL_GHAMS_2015_5282.xml,{'De gronden van de beslissing': 'Nu van de zi...,{'het verloop van het geding': '1.1 In het ver...
2,ECLI_NL_GHAMS_2016_4005,ECLI_NL_GHAMS_2016_4005.xml,{'De feiten': '2.1. Partijen zijn op 6 augustu...,{'het geding in hoger beroep': '1.1. Appellant...
3,ECLI_NL_GHAMS_2017_3990,ECLI_NL_GHAMS_2017_3990.xml,{'Feiten': '2.1. De rechtbank heeft de volgend...,{'ontstaan en loop van het geding': '1.1. De h...
4,ECLI_NL_GHAMS_2017_4813,ECLI_NL_GHAMS_2017_4813.xml,{'Onderzoek van de zaak': 'Dit arrest is gewez...,{}
...,...,...,...,...
95,ECLI_NL_RBZWB_2015_638,ECLI_NL_RBZWB_2015_638.xml,{'Overwegingen': '1. Op grond van de stukken e...,{'procesverloop': 'Eiser heeft beroep ingestel...
96,ECLI_NL_RBZWB_2015_6953,ECLI_NL_RBZWB_2015_6953.xml,{'De feiten': '2.1. [eiseres] en [naam ex-part...,{'de procedure': '1.1. Het verloop van de proc...
97,ECLI_NL_RBZWB_2019_5711,ECLI_NL_RBZWB_2019_5711.xml,{'Onderzoek van de zaak': 'De zaak is inhoudel...,{'de wettelijke voorschriften': 'De beslissing...
98,ECLI_NL_RBZWB_2020_2644,ECLI_NL_RBZWB_2020_2644.xml,"{'de gemeente GEMEENTE NOORD-BEVELAND,': 'geve...",{'de procedure': '1.1. Het verloop van de proc...


### Step 3: Matching the remaining section titles/headers to predefined groups for consistency. 

In [10]:
SEED_WORDS_LIST = {
    "Context": [
        "context_overig", "voorvragen", "de voorvragen", "inleiding", "grondslag en inhoud van het eab",
        "onderzoek van de zaak", "het onderzoek ter terechtzitting", "onderzoek ter terechtzitting", "gronden",
        "het onderzoek op de terechtzitting", "onderzoek op de terechtzitting", "de kern van de zaak",
        "waar gaat deze zaak over?", "inleiding en samenvatting", "waar gaat het over?", "het bewijs",
        "waar gaat de zaak over?"
    ],
    "Feiten": [
        "feiten", "de feiten", "feiten", "de zaak in het kort", "de vaststaande feiten", "vaststaande feiten",
        "uitgangspunten en feiten", "feiten en procesverloop", "feitelijke achtergrond", "uitgangspunten in cassatie"
    ],
    "Beoordeling": [
        "beoordeling_door_rechter", "de beoordeling", "waardering van het bewijs", "de motivering van de beslissing",
        "beoordeling", "bewezenverklaring", "de beoordeling van het bewijs", "de verdere beoordeling",
        "beoordeling van het geschil", "de strafbaarheid van de verdachte", "strafbaarheid van verdachte",
        "de strafbaarheid van verdachte", "de strafbaarheid", "beoordeling door de rechtbank", "strafbaarheid",
        "beoordeling van het middel", "strafbaarheid van de feiten", "de strafbaarheid van het bewezenverklaarde",
        "conclusie en gevolgen", "de bewezenverklaring", "de strafbaarheid van de feiten",
        "de strafbaarheid van het bewezen verklaarde", "beoordeling van het cassatiemiddel",
        "beoordeling van de middelen", "bespreking van het cassatiemiddel", "de bewijsmotivering",
        "beoordeling van het bewijs", "strafbaarheid verdachte", "de motivering van de beslissing in hoger beroep",
        "beoordeling van de ontvankelijkheid van het beroep in cassatie", "de bewijsbeslissing",
        "de kwalificatie van het bewezenverklaarde", "strafbaarheid van de verdachte", "de gronden van de beslissing",
        "motivering van de straf", "de strafbaarheid van het feit", "de beoordeling van het geschil",
        "strafbaarheid van het feit", "motivering straf", "motivering van de sanctie", "het oordeel van de rechtbank",
        "de beoordeling in het incident", "beoordeling van de klachten", "de beoordeling in hoger beroep",
        "overwegingen", "kwalificatie en strafbaarheid van de feiten", "beoordeling van de cassatiemiddelen",
        "beoordeling van het eerste cassatiemiddel", "strafbaarheid feiten",
        "de overwegingen ten aanzien van straf en/of maatregel", "beoordeling van het tweede cassatiemiddel",
        "beoordeling in hoger beroep", "strafbaarheid feit", "motivering van de straffen en maatregelen",
        "kwalificatie en strafbaarheid van het feit", "de beoordeling in conventie en in reconventie",
        "het geschil in reconventie", "de straf en/of de maatregel", "bewijs", "het eerste middel",
        "de beoordeling van de civiele vordering"
    ],
    "Beslissing": [
        "beslissing", "de beslissing", "beslissing", "slotsom", "de uitspraak", "de slotsom", "de strafoplegging",
        "conclusie", "oplegging van straf", "de op te leggen straf of maatregel", "vrijspraak", ".beslissing",
        ". beslissing", "de straf"
    ],
    "Proceshandelingen partijen": [
        "proceshandelingen_partijen", "het geschil", "tenlastelegging", "de tenlastelegging", "de vordering",
        "het verweer", "het verzoek", "geschil", "geding in cassatie", "eis officier van justitie",
        "de vordering en het verweer", "de inhoud van de tenlastelegging", "het wrakingsverzoek",
        "geschil in hoger beroep", "het verzoek en het verweer", "geschil en conclusies van partijen",
        "het cassatieberoep", "het geschil in conventie", "de standpunten", "standpunten van partijen",
        "het standpunt van de officier van justitie", "de vordering tot tenuitvoerlegging"
    ]
}

In [11]:
import re
import pandas as pd
from collections import defaultdict

# Try RapidFuzz; fallback to difflib if not installed
try:
    from rapidfuzz import fuzz
    _HAS_RAPIDFUZZ = True
except ImportError:
    import difflib
    _HAS_RAPIDFUZZ = False


# -------------------------
# Normalization + tokens
# -------------------------
DUTCH_STOP_TOKENS = {
    "de", "het", "een", "en", "van", "in", "op", "aan", "bij", "voor", "met", "tot", "uit", "door",
    "over", "onder", "naar", "als", "is", "zijn", "wordt", "werden", "was", "dat", "die", "dit",
    "te", "ten", "ter", "der", "den", "of", "maar", "ook", "nog", "dan", "wel", "niet"
}

def normalize_header(s: str) -> str:
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s).strip()
    s = s.strip(" .:-–—\t")
    return s

def tokenize_for_match(s: str):
    s = normalize_header(s)
    toks = re.findall(r"[a-z0-9]+", s)
    toks = [t for t in toks if t not in DUTCH_STOP_TOKENS]
    return toks


# -------------------------
# Seed structures
# -------------------------
def build_seed_structures(seed_dict):
    """
    seed_dict: {group_name: [phrases...]}

    Returns:
      exact_map: norm_phrase -> group_name
      group_phrase_tokens: group_name -> list of (norm_phrase, token_set)
      all_seed_phrases: list of (group_name, norm_phrase)
      seed_phrase_tokens_flat: list of (group_name, norm_phrase, token_set)
    """
    exact_map = {}
    group_phrase_tokens = defaultdict(list)
    all_seed_phrases = []
    seed_phrase_tokens_flat = []

    for group_name, phrases in seed_dict.items():
        if not isinstance(phrases, (list, tuple)):
            continue

        for phrase in phrases:
            if not isinstance(phrase, str):
                continue

            norm_phrase = normalize_header(phrase)
            if not norm_phrase:
                continue

            # exact map first occurrence wins
            if norm_phrase not in exact_map:
                exact_map[norm_phrase] = group_name

            tok_set = set(tokenize_for_match(phrase))
            if tok_set:
                group_phrase_tokens[group_name].append((norm_phrase, tok_set))
                seed_phrase_tokens_flat.append((group_name, norm_phrase, tok_set))

            all_seed_phrases.append((group_name, norm_phrase))

    return exact_map, group_phrase_tokens, all_seed_phrases, seed_phrase_tokens_flat


EXACT_SEED_MAP, GROUP_PHRASE_TOKENS, ALL_SEED_PHRASES, SEED_PHRASE_TOKENS_FLAT = build_seed_structures(SEED_WORDS_LIST)


# -------------------------
# Matching: exact → token overlap → fuzzy
# -------------------------
def token_overlap_match(title: str, min_overlap_tokens: int = 2, min_jaccard: float = 0.34):
    title_tokens = set(tokenize_for_match(title))
    if not title_tokens:
        return (False, None, {})

    best = None

    for group_name, phrase_list in GROUP_PHRASE_TOKENS.items():
        for seed_phrase, seed_tokens in phrase_list:
            inter = title_tokens & seed_tokens
            overlap = len(inter)

            if overlap == 0:
                continue

            union = len(title_tokens | seed_tokens)
            jacc = overlap / union if union else 0.0

            cand = (overlap, jacc, group_name, seed_phrase, inter)

            if best is None or (cand[0], cand[1]) > (best[0], best[1]):
                best = cand

    if best is None:
        return (False, None, {})

    overlap, jacc, group_name, seed_phrase, inter = best

    if overlap >= min_overlap_tokens and jacc >= min_jaccard:
        return (True, group_name, {
            "best_seed": seed_phrase,
            "overlap": overlap,
            "jaccard": jacc,
            "shared_tokens": sorted(list(inter))
        })

    return (False, None, {
        "best_seed": seed_phrase,
        "overlap": overlap,
        "jaccard": jacc,
        "shared_tokens": sorted(list(inter))
    })


def _fuzzy_score(a: str, b: str) -> float:
    if _HAS_RAPIDFUZZ:
        return float(fuzz.token_set_ratio(a, b))
    else:
        return float(difflib.SequenceMatcher(None, a, b).ratio() * 100.0)


def fuzzy_match(
    title: str,
    min_score: float = 90.0,
    min_title_len: int = 8,
    require_token_overlap: bool = True,
    min_overlap_for_fuzzy: int = 1
):
    norm_title = normalize_header(title)

    if len(norm_title) < min_title_len:
        return (False, None, {
            "reason": "title_too_short",
            "norm_title": norm_title
        })

    title_tokens = set(tokenize_for_match(title))

    best = None

    for group_name, seed_phrase in ALL_SEED_PHRASES:
        score = _fuzzy_score(norm_title, seed_phrase)

        if best is None or score > best[0]:
            best = (score, group_name, seed_phrase)

    if best is None:
        return (False, None, {
            "reason": "no_seeds",
            "norm_title": norm_title
        })

    score, group_name, seed_phrase = best

    debug = {
        "norm_title": norm_title,
        "best_seed": seed_phrase,
        "score": score
    }

    if score < min_score:
        debug["reason"] = "score_below_threshold"
        return (False, None, debug)

    if require_token_overlap:
        seed_tokens = set(tokenize_for_match(seed_phrase))
        inter = title_tokens & seed_tokens

        debug["shared_tokens"] = sorted(list(inter))
        debug["overlap"] = len(inter)

        if len(inter) < min_overlap_for_fuzzy:
            debug["reason"] = "no_token_overlap_with_best_seed"
            return (False, None, debug)

    return (True, group_name, debug)


def match_group_cascade(
    title: str,
    token_min_overlap: int = 2,
    token_min_jaccard: float = 0.34,
    fuzzy_min_score: float = 90.0,
    fuzzy_min_title_len: int = 8,
    fuzzy_require_token_overlap: bool = True,
    fuzzy_min_overlap: int = 1
):
    norm_title = normalize_header(title)

    if not norm_title:
        return (False, None, norm_title, None, {})

    # 1) Exact
    group = EXACT_SEED_MAP.get(norm_title)

    if group is not None:
        return (True, group, norm_title, "exact", {
            "norm_title": norm_title
        })

    # 2) Token overlap
    ok2, group2, dbg2 = token_overlap_match(
        title,
        min_overlap_tokens=token_min_overlap,
        min_jaccard=token_min_jaccard
    )

    if ok2:
        dbg2["norm_title"] = norm_title
        return (True, group2, norm_title, "token_overlap", dbg2)

    # 3) Fuzzy
    ok3, group3, dbg3 = fuzzy_match(
        title,
        min_score=fuzzy_min_score,
        min_title_len=fuzzy_min_title_len,
        require_token_overlap=fuzzy_require_token_overlap,
        min_overlap_for_fuzzy=fuzzy_min_overlap
    )

    if ok3:
        return (True, group3, norm_title, "fuzzy", dbg3)

    return (False, None, norm_title, None, {
        "norm_title": norm_title
    })


# -------------------------
# Main matching loop + audits + unmatched
# -------------------------
filtered_rows = []

token_overlap_audit_rows = []
fuzzy_audit_rows = []

unmatched_by_case = defaultdict(list)
unmatched_rows = []

for _, row in df_final.iterrows():
    case_name = row["case_name"]
    extracted_items = row["extracted_data"]

    if not isinstance(extracted_items, dict):
        continue

    kept = []

    for title, text in extracted_items.items():
        norm_title = normalize_header(title)

        ok, group_name, norm_title, method, debug = match_group_cascade(
            title,
            token_min_overlap=2,
            token_min_jaccard=0.34,
            fuzzy_min_score=80.0,
            fuzzy_min_title_len=8,
            fuzzy_require_token_overlap=True,
            fuzzy_min_overlap=1
        )

        if ok:
            kept.append({
                "title": title,
                "text": text,
                "hdr_group": group_name,
                "norm_title": norm_title,
                "match_method": method,
                "match_debug": debug
            })

            if method == "token_overlap":
                token_overlap_audit_rows.append({
                    "case_name": case_name,
                    "original_title": title,
                    "normalized_title": norm_title,
                    "assigned_group": group_name,
                    "best_seed": debug.get("best_seed"),
                    "shared_tokens": debug.get("shared_tokens"),
                    "overlap": debug.get("overlap"),
                    "jaccard": debug.get("jaccard"),
                    "section_text_snippet": (text or "")[:300]
                })

            elif method == "fuzzy":
                fuzzy_audit_rows.append({
                    "case_name": case_name,
                    "original_title": title,
                    "normalized_title": norm_title,
                    "assigned_group": group_name,
                    "best_seed": debug.get("best_seed"),
                    "fuzzy_score": debug.get("score"),
                    "shared_tokens": debug.get("shared_tokens"),
                    "overlap": debug.get("overlap"),
                    "section_text_snippet": (text or "")[:300],
                    "used_rapidfuzz": _HAS_RAPIDFUZZ
                })

        else:
            if title and norm_title:
                entry = {
                    "title": title,
                    "norm_title": norm_title,
                    "section_text_snippet": (text or "")[:300],
                }

                unmatched_by_case[case_name].append(entry)
                unmatched_rows.append({
                    "case_name": case_name,
                    **entry
                })

    if kept:
        title_text_dict = {
            k["title"]: k["text"]
            for k in kept
            if k["title"] and k["text"]
        }

        filtered_rows.append({
            "case_name": case_name,
            "extracted_data": title_text_dict,
            "matched_headers": [k["title"] for k in kept],
            "matched_groups": [k["hdr_group"] for k in kept],
            "matched_norm_titles": [k["norm_title"] for k in kept],
            "match_methods": [k["match_method"] for k in kept],
        })


df_filtered_cascade = pd.DataFrame(filtered_rows).reset_index(drop=True)
df_token_overlap_audit = pd.DataFrame(token_overlap_audit_rows)
df_fuzzy_audit = pd.DataFrame(fuzzy_audit_rows)
df_unmatched = pd.DataFrame(unmatched_rows).reset_index(drop=True)

print("Cases with ≥1 unmatched header:", len(unmatched_by_case))
print("Total unmatched headers:", len(df_unmatched))

df_filtered_cascade, df_token_overlap_audit, df_fuzzy_audit, df_unmatched

Cases with ≥1 unmatched header: 62
Total unmatched headers: 121


(                  case_name  \
 0   ECLI_NL_GHAMS_2015_2960   
 1   ECLI_NL_GHAMS_2015_5282   
 2   ECLI_NL_GHAMS_2016_4005   
 3   ECLI_NL_GHAMS_2017_3990   
 4   ECLI_NL_GHAMS_2017_4813   
 ..                      ...   
 95   ECLI_NL_RBZWB_2015_638   
 96  ECLI_NL_RBZWB_2015_6953   
 97  ECLI_NL_RBZWB_2019_5711   
 98  ECLI_NL_RBZWB_2020_2644   
 99  ECLI_NL_RBZWB_2020_4123   
 
                                        extracted_data  \
 0   {'Feiten': 'De kantonrechter heeft in het best...   
 1   {'De gronden van de beslissing': 'Nu van de zi...   
 2   {'De feiten': '2.1. Partijen zijn op 6 augustu...   
 3   {'Feiten': '2.1. De rechtbank heeft de volgend...   
 4   {'Onderzoek van de zaak': 'Dit arrest is gewez...   
 ..                                                ...   
 95  {'Overwegingen': '1.	Op grond van de stukken e...   
 96  {'De feiten': '2.1. [eiseres] en [naam ex-part...   
 97  {'Onderzoek van de zaak': 'De zaak is inhoudel...   
 98  {'De feiten': '2.1. [gedaagde

### Step 4: Combine all the corresponding section data into one text so that it can be passed later to the clean and segment it into sentences. 

In [12]:
df_final = df_filtered_cascade.copy()
df_final['data']=df_final['extracted_data'].apply(lambda d: ''.join(list(d.values())) if isinstance(d, dict) else None)
df_final

,case_name,extracted_data,matched_headers,matched_groups,matched_norm_titles,match_methods,data
0,ECLI_NL_GHAMS_2015_2960,{'Feiten': 'De kantonrechter heeft in het best...,"[Feiten, Beoordeling, Beslissing]","[Feiten, Beoordeling, Beslissing]","[feiten, beoordeling, beslissing]","[exact, exact, exact]",De kantonrechter heeft in het bestreden vonnis...
1,ECLI_NL_GHAMS_2015_5282,{'De gronden van de beslissing': 'Nu van de zi...,"[De gronden van de beslissing, De beslissing]","[Beoordeling, Beslissing]","[de gronden van de beslissing, de beslissing]","[exact, exact]",Nu van de zijde van de partijen geen bezwaren ...
2,ECLI_NL_GHAMS_2016_4005,{'De feiten': '2.1. Partijen zijn op 6 augustu...,"[De feiten, Het geschil in hoger beroep, Beoor...","[Feiten, Proceshandelingen partijen, Beoordeli...","[de feiten, het geschil in hoger beroep, beoor...","[exact, token_overlap, token_overlap, exact]",2.1. Partijen zijn op 6 augustus 2008 te Al Ho...
3,ECLI_NL_GHAMS_2017_3990,{'Feiten': '2.1. De rechtbank heeft de volgend...,"[Feiten, Geschil in hoger beroep, Beoordeling ...","[Feiten, Proceshandelingen partijen, Beoordeli...","[feiten, geschil in hoger beroep, beoordeling ...","[exact, exact, exact, fuzzy]",2.1. De rechtbank heeft de volgende feiten vas...
4,ECLI_NL_GHAMS_2017_4813,{'Onderzoek van de zaak': 'Dit arrest is gewez...,"[Onderzoek van de zaak, Tenlastelegging, Vorde...","[Context, Proceshandelingen partijen, Procesha...","[onderzoek van de zaak, tenlastelegging, vorde...","[exact, exact, fuzzy, exact, token_overlap, ex...",Dit arrest is gewezen naar aanleiding van het ...
...,...,...,...,...,...,...,...
95,ECLI_NL_RBZWB_2015_638,{'Overwegingen': '1. Op grond van de stukken e...,"[Overwegingen, Beslissing]","[Beoordeling, Beslissing]","[overwegingen, beslissing]","[exact, exact]",1.\tOp grond van de stukken en de behandeling ...
96,ECLI_NL_RBZWB_2015_6953,{'De feiten': '2.1. [eiseres] en [naam ex-part...,"[De feiten, Het geschil, De beoordeling, De be...","[Feiten, Proceshandelingen partijen, Beoordeli...","[de feiten, het geschil, de beoordeling, de be...","[exact, exact, exact, exact]",2.1. [eiseres] en [naam ex-partner] zijn gehuw...
97,ECLI_NL_RBZWB_2019_5711,{'Onderzoek van de zaak': 'De zaak is inhoudel...,"[Onderzoek van de zaak, De tenlastelegging, De...","[Context, Proceshandelingen partijen, Context,...","[onderzoek van de zaak, de tenlastelegging, de...","[exact, exact, exact, exact, exact, exact, exact]",De zaak is inhoudelijk behandeld op de zitting...
98,ECLI_NL_RBZWB_2020_2644,{'De feiten': '2.1. [gedaagde 2] is sinds 24 j...,"[De feiten, De beoordeling, De beslissing]","[Feiten, Beoordeling, Beslissing]","[de feiten, de beoordeling, de beslissing]","[exact, exact, exact]",2.1. [gedaagde 2] is sinds 24 juni 2005 eigena...


In [13]:
import re

# Basic cleaning functions for money, dates, and person names from the text. 
# It replaces them with placeholders like [geld], [datum], and [naam] respectively.

def replace_money(text):
    if pd.isna(text):
        return text
    text = str(text)
    # covers € / eur / euro followed by numbers, commas, dots, or spaces
    text = re.sub(r"(€|eur|euro)\s?[\d\.\,]+", "[geld]", text, flags=re.IGNORECASE)
    return text

def replace_dates_by_month_window(text):
    if pd.isna(text):
        return text
    text = str(text)
    months = [
        "januari","februari","maart","april","mei","juni","juli",
        "augustus","september","oktober","november","december",
        "jan","feb","mrt","apr","jun","jul","aug","sep","okt","nov","dec"
    ]
    pattern = r"(?:\b\S+\s)?\b(" + "|".join(months) + r")\b(?:\s\S+)?"
    text = re.sub(pattern, "[datum]", text, flags=re.IGNORECASE)
    return text

import re
import pandas as pd

# Compile once for speed
PARTICLES = r"(?:van|de|den|der|het|ten|ter|te|la|le|l'|d')"
SURNAME = r"[A-Z][A-Za-zÀ-ÖØ-öø-ÿ\-']+"
INITIALS_DOTTED = r"(?:[A-Z]\.\s*){1,5}"
HONORIFICS = r"(?:mr|mrs|mevr|mw|meneer|dr|drs|prof|prof\.dr)\.?"

# 1) Honorific + initials + surname  (e.g., "mr. P. Oudkerk", "MR A.B. de Vries")
pat_honorific_initials = re.compile(
    rf"\b{HONORIFICS}\s+{INITIALS_DOTTED}(?:{PARTICLES}\s+)?{SURNAME}\b",
    flags=re.IGNORECASE
)

# 2) Initials + surname (no honorific)  (e.g., "J.E. Molenaar", "C.M. Aarts")
pat_initials_surname = re.compile(
    rf"\b{INITIALS_DOTTED}(?:{PARTICLES}\s+)?{SURNAME}\b"
)

# 3) Optional broader form: Firstname + surname (riskier — leave OFF by default)
#    e.g., "Jan de Vries" | enable by setting BROAD_FIRSTNAME=True
BROAD_FIRSTNAME = False
pat_firstname_surname = re.compile(
    rf"\b[A-Z][a-zà-öø-ÿ]{{2,}}\s+(?:{PARTICLES}\s+)?{SURNAME}\b"
)

def replace_person_names(text, broad_firstname=BROAD_FIRSTNAME):
    if pd.isna(text): 
        return text
    s = str(text)

    # Protect placeholders like [appellante], [geïntimeerde], [datum], [geld]
    protected = {}
    def _protect(m):
        key = f"§§§{len(protected)}§§§"
        protected[key] = m.group(0)
        return key
    s = re.sub(r"\[[^\]\n]{1,50}\]", _protect, s)

    s = pat_honorific_initials.sub("[naam]", s)
    s = pat_initials_surname.sub("[naam]", s)
    if broad_firstname:
        s = pat_firstname_surname.sub("[naam]", s)

    # Restore placeholders
    for k, v in protected.items():
        s = s.replace(k, v)
    return s

# EXAMPLE: apply to your dataframe
# df = pd.read_csv("your_dataset.csv")

# print(df[["data_clean"]].head(10))





df_final["data_clean"] = df_final["data"].apply(replace_money)
df_final["data_clean"] = df_final["data_clean"].apply(replace_dates_by_month_window)
df_final["data_clean"] = df_final["data_clean"].astype(str).map(replace_person_names)

#df_final["data_clean"] = df_final["data_clean"].apply(replace_dates)
df_final.data_clean[0]


'De kantonrechter heeft in het bestreden vonnis onder het kopje “De feiten” de feiten vastgesteld die zij tot uitgangspunt heeft genomen. Deze feiten zijn in hoger beroep niet in geschil en dienen derhalve ook het hof als uitgangspunt.3.1. Op [datum] heeft [geïntimeerde] een bedrag van [geld]- ter leen verstrekt aan [appellante] in verband met het voornemen van [appellante] een onderneming (verkoop van voedingssupplementen) te beginnen. Op [datum] heeft [appellante] een schriftelijke verklaring opgesteld, die beide partijen hebben ondertekend. Deze verklaring houdt onder meer in: “Dit bedrag wordt in gedeelten terugbetaald; wanneer er door deze promotie, inschrijvingen en verdiensten in welke vorm dan ook; ontvangen worden; gaat daar zo snel mogelijk het grootste deel direct van terug/retour naar de rekening van dhr. [geïntimeerde] (…) Zoals afgesproken; zo ook: zou er een grote doos met voedingssupplementen ter waarde van [geld]= bij dhr. v. Diest terecht komen voor eigen gebruik (…) 

### Step 5: Sentence Splitter

In [14]:
### Split the text data into sentences ###
from sentence_splitter import sentence_splitter
ss = sentence_splitter()


df = df_final[['case_name','data_clean']].rename(columns={"data_clean":"text"})
df['label'] = ["None" for _ in range(len(df))]
text_col = "text"  #
# keep everything else as metadata
meta_cols = ["case_name"]
xml_data = ss.dataframe_to_sentences(df, text_col="text", label_col="label", meta_cols=meta_cols)
#xml_data.to_csv(r"final/annotations_sentences_clean_v4.csv", index=False)
xml_data.head()



,block_row,sent_index,sent_text,sent_len,label,case_name
0,1,1,De kantonrechter heeft in het bestreden vonnis...,136,None,ECLI_NL_GHAMS_2015_2960
1,1,2,Deze feiten zijn in hoger beroep niet in gesch...,97,None,ECLI_NL_GHAMS_2015_2960
2,1,3,Op [datum] heeft [geïntimeerde] een bedrag van...,200,None,ECLI_NL_GHAMS_2015_2960
3,1,4,Op [datum] heeft [appellante] een schriftelijk...,106,None,ECLI_NL_GHAMS_2015_2960
4,1,5,Deze verklaring houdt onder meer in: “Dit bedr...,278,None,ECLI_NL_GHAMS_2015_2960
